# Assignment 16: Preference Tuning with DPO

Fill in every `# TODO` and answer every question in the **Your answer** cells. Run all cells before submitting.

## Setup

Put `tiny_gpt.py`, your `instruct_model.pt` from Assignment 15 (or the provided one) and `tinystories.txt` in the same folder as this notebook.

In [ ]:
import re
import copy
import random
import torch
import matplotlib.pyplot as plt
from torch.nn import functional as F
from tiny_gpt import encode, generate, estimate_loss, load_model, save_model

random.seed(0)
torch.manual_seed(0)
TOPICS = ["dog", "cat", "bird", "bear", "fish", "frog", "ball", "cake", "car", "tree"]

def prompt_for(topic):
    return f"Topic: {topic}\nStory: "

def mentions(topic, text):
    return re.search(rf"\b{topic}s?\b", text, re.IGNORECASE) is not None

def on_topic_rate(model, repeats=10, seed=1):
    torch.manual_seed(seed)
    return {t: sum(mentions(t, generate(model, prompt_for(t), 120)) for _ in range(repeats)) / repeats
            for t in TOPICS}

instruct = load_model("instruct_model.pt")
instruct_rates = on_topic_rate(instruct)
print(f"Instruction-tuned model: {sum(instruct_rates.values()) / len(TOPICS):.0%} on topic")

## Step 1. Preferences instead of correct answers

Instruction tuning needs a **correct response** for every prompt. For many tasks it is much easier to **compare** two responses and say which one is better. That is what preference tuning learns from.

In real systems the comparisons come from people (or from a reward model trained on people's comparisons). Here we use a simple automatic judge: **a response is better if it is about the requested topic.**

### Question 1. Collect preference pairs

For each prompt, sample 4 responses from the instruction-tuned model. If at least one is on topic and at least one is not, make a pair `(prompt, chosen, rejected)` with an on-topic response as `chosen` and an off-topic response as `rejected`.

Complete the loop below. Use 300 prompts, cycling through the topics (about 2 minutes). Why do we skip prompts where all 4 responses are on topic, or all are off topic?

In [ ]:
torch.manual_seed(2)
pairs = []
for i in range(300):
    topic = TOPICS[i % len(TOPICS)]
    prompt = prompt_for(topic)
    responses = [generate(instruct, prompt, 120) + "\n\n" for _ in range(4)]
    # TODO: split the responses into on-topic (good) and off-topic (bad); if both exist, append (prompt, good[0], bad[0])

print("Number of preference pairs:", len(pairs))
prompt, chosen, rejected = pairs[0]
print(f"\n{prompt}\n  chosen:   {chosen.strip()}\n  rejected: {rejected.strip()}")

**Your answer:** *Write your answer here.*

## Step 2. Direct Preference Optimization (DPO)

DPO pushes up the probability of the chosen response and pushes down the probability of the rejected one, **relative to a frozen copy of the starting model** (the *reference* model):

$$
\text{loss} = -\log \sigma\Big(\beta \big[(\log \pi(\text{chosen}) - \log \pi_{\text{ref}}(\text{chosen})) - (\log \pi(\text{rejected}) - \log \pi_{\text{ref}}(\text{rejected}))\big]\Big)
$$

where $\log \pi(\text{response})$ is the log-probability that the model being trained gives to the whole response after the prompt, $\sigma$ is the sigmoid function and $\beta$ (we use 0.1) controls how far the model may move from the reference.

`response_logprob` computes $\log \pi(\text{response})$: the sum, over the characters of the response, of the log-probability the model gives to each actual next character.

In [ ]:
def response_logprob(model, prompt, response):
    ids = torch.tensor([encode(prompt + response)])[:, :model.context_size + 1]   # must fit in the context
    logits, _ = model(ids[:, :-1])
    logprobs = F.log_softmax(logits, dim=-1).gather(-1, ids[:, 1:].unsqueeze(-1)).squeeze(-1)
    n = ids.shape[1] - len(encode(prompt))     # number of response characters that fit
    return logprobs[0, -n:].sum()              # only the response's characters count

### Question 2. The DPO loss

a. Complete `dpo_loss`. *Hint:* `F.logsigmoid`.

b. At the start of training the model being trained equals the reference. What is the loss then? (Compute it by hand, then check with the code.)

c. In words: what must happen to the two log-probabilities for the loss to go down?

In [ ]:
def dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta=0.1):
    # TODO: the DPO loss from the formula above

x = torch.tensor(-50.0)
print("Loss when the model equals the reference:", dpo_loss(x, x - 10, x, x - 10).item())

**Your answer:** *Write your answer here.*

### Question 3. Train with DPO

Complete the training loop: for each pair compute the four log-probabilities (the reference model's without gradients), the loss, and take an optimizer step. Train for 2 passes over the pairs (about 2–4 minutes).

We also track the **preference accuracy**: the fraction of pairs where the model (relative to the reference) already prefers the chosen response.

In [ ]:
reference = copy.deepcopy(instruct).eval()
for p in reference.parameters():
    p.requires_grad_(False)
policy = copy.deepcopy(instruct)
optimizer = torch.optim.AdamW(policy.parameters(), lr=5e-5)

for epoch in range(2):
    random.shuffle(pairs)
    total_loss = correct = 0
    for prompt, chosen, rejected in pairs:
        # TODO: compute the loss for this pair and take one optimizer step
        total_loss += loss.item()
        correct += (pol_c - ref_c) > (pol_r - ref_r)
    print(f"epoch {epoch + 1}: DPO loss {total_loss / len(pairs):.3f}   preference accuracy {int(correct) / len(pairs):.2f}")

save_model(policy, "dpo_model.pt")

### Question 4. Did preference tuning help?

Compute the on-topic rate of the DPO model and plot the three stages (use your base-model rate from Assignment 15; the provided base model scores about 3%). Show stories for 3 topics.

In [ ]:
# TODO: compute dpo_rates
stages = {"base (A14)": 0.03,
          "instruction-tuned (A15)": sum(instruct_rates.values()) / len(TOPICS),
          "DPO (A16)": sum(dpo_rates.values()) / len(TOPICS)}
for name, rate in stages.items():
    print(f"{name:<26} {rate:.0%} on topic")
plt.bar(stages.keys(), stages.values()); plt.ylabel("on-topic rate"); plt.ylim(0, 1); plt.show()

torch.manual_seed(3)
for t in ["dog", "frog", "cake"]:
    print(f"{prompt_for(t)}{generate(policy, prompt_for(t), 120)}\n")

**Your answer:** *Write your answer here.*

### Question 5. What did it cost?

Preference tuning optimizes **one** thing (our judge). Did the model get worse at anything else?

a. Compare the loss of the instruction-tuned and the DPO model on plain stories.

b. Read 10 DPO stories. Has the model found a shortcut to please the judge (for example, repeating the topic word, or always starting the same way)? This is called **reward hacking**.

c. What is the reference model for? What would happen with a much larger $\beta$ or learning rate?

In [ ]:
with open("tinystories.txt", encoding="utf-8") as f:
    plain = f.read()
plain_val = torch.tensor(encode(plain[int(0.9 * len(plain)):]))
torch.manual_seed(0)
print(f"Loss on plain stories: instruction-tuned {estimate_loss(instruct, plain_val):.3f}, DPO {estimate_loss(policy, plain_val):.3f}")

torch.manual_seed(5)
for i in range(10):
    t = TOPICS[i]
    print(f"[{t}] {generate(policy, prompt_for(t), 120)}")

**Your answer:** *Write your answer here.*

### Question 6. Reflection

In 4–5 sentences: how is what you did similar to the preference tuning of real chat models (RLHF or DPO), and how is it different? Why can't the judge used by real systems be a simple rule like ours?

**Your answer:** *Write your answer here.*